# RSHazeNet - Remote Sensing Image Dehazing (VLM-guided + Resume)

This notebook trains and evaluates RSHazeNet on **all fog levels** of two remote sensing dehazing datasets:

**Datasets**:
- **SateHaze1k** (Kaggle: `xuxingxing233/satehaze1k`) — thin, moderate, thick
- **RRSHID** (GitHub release) — thin_fog, moderate_fog, thick_fog

**Model**: RSHazeNet

**Features**:
- **VLM-guided dehazing**: a VLM captions each image's haze level offline (cached), then a frozen CLIP text encoder conditions the bottleneck via cross-attention (`--caption`).
- **Resume training**: continue from a full-state checkpoint with `--resume`.

**Metrics**: PSNR, SSIM, SAM, ERGAS (validation) + comprehensive evaluation

In [ ]:
#@title 1. Install Dependencies
!pip install -q timm pytorch-msssim lpips scikit-image thop gdown transformers accelerate

In [ ]:
#@title 2. Setup: Output Directories & Clone Repository
import os

# Output directories (persisted in /kaggle/working/)
RESULTS_DIR = '/kaggle/working/RSHazeNet_Results'
CAPTIONS_DIR = '/kaggle/working/captions'  # writable location for caption JSONs
os.makedirs(f'{RESULTS_DIR}/pretrained_results', exist_ok=True)
os.makedirs(f'{RESULTS_DIR}/trained_results', exist_ok=True)
os.makedirs(f'{RESULTS_DIR}/weights', exist_ok=True)
os.makedirs(CAPTIONS_DIR, exist_ok=True)

def caption_path(input_dir):
    """Map any hazy image dir to its captions.json under CAPTIONS_DIR.
    Needed because /kaggle/input/ is read-only.
    E.g. /kaggle/input/satehaze1k/Haze1k_thick/train/hazy
      -> /kaggle/working/captions/satehaze1k_Haze1k_thick_train_hazy/captions.json"""
    key = input_dir.replace('/', '_').strip('_')
    out_dir = os.path.join(CAPTIONS_DIR, key)
    os.makedirs(out_dir, exist_ok=True)
    return os.path.join(out_dir, 'captions.json')

# Clone the repository
!git clone https://github.com/sumire25/RSHazeNet.git /kaggle/working/RSHazeNet
%cd /kaggle/working/RSHazeNet
!ls

In [ ]:
#@title 3. Verify SateHaze1k Dataset (Kaggle Input)
# SateHaze1k is attached as a Kaggle dataset: xuxingxing233/satehaze1k
# Expected structure: /kaggle/input/satehaze1k/Haze1k_{level}/{train,test}/{GT,hazy}/

SATEHAZE_BASE = '/kaggle/input/datasets/xuxingxing233/satehaze1k'
SATEHAZE_LEVELS = ['Haze1k_thin', 'Haze1k_moderate', 'Haze1k_thick']

print('=== SateHaze1k Dataset Structure ===')
for level in SATEHAZE_LEVELS:
    level_path = os.path.join(SATEHAZE_BASE, level)
    if os.path.isdir(level_path):
        for split in ['train', 'test']:
            split_path = os.path.join(level_path, split)
            if os.path.isdir(split_path):
                subfolders = os.listdir(split_path)
                counts = {sf: len(os.listdir(os.path.join(split_path, sf))) 
                          for sf in subfolders if os.path.isdir(os.path.join(split_path, sf))}
                print(f'  {level}/{split}: {counts}')
    else:
        print(f'  WARNING: {level_path} not found!')
# Note: SateHaze1k has no separate 'valid' split.
# We use the 'test' split as validation during training.

In [ ]:
#@title 3b. Download & Extract RRSHID Dataset
import os, glob

RRSHID_ZIP = '/kaggle/working/RRSHID.zip'
RRSHID_BASE = '/kaggle/working/RRSHID'

if not os.path.isdir(RRSHID_BASE):
    !wget -q -O {RRSHID_ZIP} "https://github.com/lwCVer/RRSHID/releases/download/dataset/RRSHID.zip"
    !unzip -q {RRSHID_ZIP} -d {RRSHID_BASE}
    !rm -f {RRSHID_ZIP}
    print('RRSHID extracted.')
else:
    print('RRSHID already exists, skipping download.')

# Auto-detect GT folder name (could be 'clear', 'gt', or 'GT')
RRSHID_LEVELS = ['thin_fog', 'moderate_fog', 'thick_fog']

def find_gt_folder(base_path, level, split):
    """Find the ground-truth folder name for a given RRSHID split."""
    split_path = os.path.join(base_path, level, split)
    if not os.path.isdir(split_path):
        return None
    for candidate in ['GT', 'gt', 'clear']:
        if os.path.isdir(os.path.join(split_path, candidate)):
            return candidate
    return None

print('\n=== RRSHID Dataset Structure ===')
for level in RRSHID_LEVELS:
    level_path = os.path.join(RRSHID_BASE, level)
    if os.path.isdir(level_path):
        for split in ['train', 'val', 'test']:
            split_path = os.path.join(level_path, split)
            if os.path.isdir(split_path):
                gt_name = find_gt_folder(RRSHID_BASE, level, split)
                subfolders = os.listdir(split_path)
                counts = {sf: len(os.listdir(os.path.join(split_path, sf)))
                          for sf in subfolders if os.path.isdir(os.path.join(split_path, sf))}
                print(f'  {level}/{split}: {counts}  (GT folder: {gt_name})')
    else:
        print(f'  WARNING: {level_path} not found!')
        # Try to find it nested (some zips have an extra directory level)
        nested = glob.glob(f'{RRSHID_BASE}/*/{level}')
        if nested:
            print(f'    Found at: {nested[0]}')
            RRSHID_BASE = os.path.dirname(nested[0])
            print(f'    Updated RRSHID_BASE to: {RRSHID_BASE}')

In [ ]:
#@title 4. Caption Dataset Haziness with VLM (Main Process)
import os
import caption  # Imports your caption.py file as a Python module
from tqdm import tqdm

# 1. Initialize the VLM exactly once for the entire session
print("Initializing VLM and allocating to GPU...")
caption_fn, cleanup = caption.build_captioner(
    vlm_model_id="Qwen/Qwen2-VL-2B-Instruct",
    api_key="",
    api="local",
    device="cuda",
    max_side=512
)
print("Model loaded successfully. Starting batch processing.\n")

def process_directory(input_dir):
    """Processes a single directory using the pre-loaded VLM."""
    if not os.path.isdir(input_dir):
        print(f'Skipping (not found): {input_dir}')
        return

    # Use your original path routing logic here
    out_dir = os.path.dirname(caption_path(input_dir))
    os.makedirs(out_dir, exist_ok=True)
    
    cache_path = os.path.join(out_dir, "captions.json")
    
    # Access the helper functions directly from your caption.py module
    cache = caption.load_existing(cache_path)
    images = caption.list_images(input_dir)
    
    todo = [f for f in images if f not in cache]
    print(f'Captioning: {input_dir} -> {out_dir}')
    print(f'Total images: {len(images)} | To caption: {len(todo)}')
    
    if not todo:
        return

    # Run inference natively in the main thread
    for i, fname in enumerate(tqdm(todo, desc="Processing")):
        ipath = os.path.join(input_dir, fname)
        try:
            text = caption_fn(ipath)
            level = caption.parse_level(text)
            cache[fname] = {"caption": text, "level": level}
        except Exception as e:
            print(f"\n[WARN] Failed on {fname}: {e}")
            
        # Checkpoint the JSON periodically
        if (i + 1) % 20 == 0 or (i + 1) == len(todo):
            caption.save_cache(cache_path, cache)

# --- SateHaze1k ---
for level in SATEHAZE_LEVELS:
    for split in ['train', 'test']:
        process_directory(f'{SATEHAZE_BASE}/{level}/{split}/hazy')

# --- RRSHID ---
for level in RRSHID_LEVELS:
    for split in ['train', 'val', 'test']:
        process_directory(f'{RRSHID_BASE}/{level}/{split}/hazy')

print('\nAll directories complete!')

# 2. Free VRAM only when the entire job is done
cleanup()

In [ ]:
#@title 5. Download Pretrained Weights
!gdown "1Leyg1sw4x48wEo5zsPKBGtVaCF5NWdY_" -O rs_haze.pth
!ls -lh rs_haze.pth

In [ ]:
#@title 6. Compute FLOPs and Parameters
import torch
from thop import profile
from model import RSHazeNet

model = RSHazeNet().cuda()
model.eval()

dummy_input = torch.randn(1, 3, 512, 512).cuda()
macs, params = profile(model, inputs=(dummy_input,))

flops_g = macs / 1e9
params_m = params / 1e6

print(f"\n--- RSHazeNet Complexity ---")
print(f"FLOPs: {flops_g:.4f} G")
print(f"Parameters: {params_m:.4f} M")

# Also measure with VLM text conditioning (dummy CLIP tokens)
text_tokens = torch.randn(1, 12, 512).cuda()
macs_c, _ = profile(model, inputs=(dummy_input, text_tokens))
print(f'Conditioned FLOPs: {macs_c/1e9:.4f} G')

del model, dummy_input, text_tokens
torch.cuda.empty_cache()

---
## Phase 1: Test Pretrained Weights

Run inference and evaluate the pretrained model on all fog levels.

In [ ]:
#@title 7. Run Inference with Pretrained Weights (All Levels)

# --- SateHaze1k (test split) ---
for level in SATEHAZE_LEVELS:
    test_hazy = f'{SATEHAZE_BASE}/{level}/test/hazy'
    test_gt   = f'{SATEHAZE_BASE}/{level}/test/GT'
    cap_test  = caption_path(test_hazy)
    result_dir = f'{RESULTS_DIR}/pretrained_results/SateHaze1k_{level}'
    os.makedirs(result_dir, exist_ok=True)
    
    if os.path.isdir(test_hazy) and os.path.isdir(test_gt):
        print(f'\n=== Inference: SateHaze1k {level} ===')
        !python infer.py \
          --test_input {test_hazy} \
          --test_target {test_gt} \
          --weights rs_haze.pth \
          --caption \
          --caption_test {cap_test} \
          --result_path {result_dir}

# --- RRSHID (test split) ---
for level in RRSHID_LEVELS:
    gt_name = find_gt_folder(RRSHID_BASE, level, 'test')
    if gt_name is None:
        print(f'Skipping RRSHID {level}/test: GT folder not found')
        continue
    test_hazy = f'{RRSHID_BASE}/{level}/test/hazy'
    test_gt   = f'{RRSHID_BASE}/{level}/test/{gt_name}'
    cap_test  = caption_path(test_hazy)
    result_dir = f'{RESULTS_DIR}/pretrained_results/RRSHID_{level}'
    os.makedirs(result_dir, exist_ok=True)
    
    if os.path.isdir(test_hazy) and os.path.isdir(test_gt):
        print(f'\n=== Inference: RRSHID {level} ===')
        !python infer.py \
          --test_input {test_hazy} \
          --test_target {test_gt} \
          --weights rs_haze.pth \
          --caption \
          --caption_test {cap_test} \
          --result_path {result_dir}

In [ ]:
#@title 8. Evaluate Pretrained Model (All Levels)

# --- SateHaze1k ---
for level in SATEHAZE_LEVELS:
    result_dir = f'{RESULTS_DIR}/pretrained_results/SateHaze1k_{level}'
    test_gt    = f'{SATEHAZE_BASE}/{level}/test/GT'
    if os.path.isdir(result_dir) and os.path.isdir(test_gt):
        print(f'\n=== Evaluation: SateHaze1k {level} ===')
        !python evaluate.py \
          --train_folder {result_dir} \
          --target_folder {test_gt}

# --- RRSHID ---
for level in RRSHID_LEVELS:
    gt_name = find_gt_folder(RRSHID_BASE, level, 'test')
    if gt_name is None:
        continue
    result_dir = f'{RESULTS_DIR}/pretrained_results/RRSHID_{level}'
    test_gt    = f'{RRSHID_BASE}/{level}/test/{gt_name}'
    if os.path.isdir(result_dir) and os.path.isdir(test_gt):
        print(f'\n=== Evaluation: RRSHID {level} ===')
        !python evaluate.py \
          --train_folder {result_dir} \
          --target_folder {test_gt}

---
## Phase 2: Train from Scratch

Train RSHazeNet (VLM-guided) on all fog levels. Each dataset/level is trained separately.

**SateHaze1k**: Uses `test` as validation (no separate valid split).  
**RRSHID**: Has a dedicated `val` split.

In [ ]:
#@title 9a. Train on SateHaze1k (All Levels)
# Train each SateHaze1k fog level. Test split is used as validation.

for level in SATEHAZE_LEVELS:
    train_hazy = f'{SATEHAZE_BASE}/{level}/train/hazy'
    train_gt   = f'{SATEHAZE_BASE}/{level}/train/GT'
    val_hazy   = f'{SATEHAZE_BASE}/{level}/test/hazy'   # test as val
    val_gt     = f'{SATEHAZE_BASE}/{level}/test/GT'      # test as val
    cap_train  = caption_path(train_hazy)
    cap_val    = caption_path(val_hazy)
    save_dir   = f'{RESULTS_DIR}/weights/SateHaze1k_{level}'
    os.makedirs(save_dir, exist_ok=True)

    if not (os.path.isdir(train_hazy) and os.path.isdir(train_gt)):
        print(f'Skipping {level}: paths not found')
        continue

    print(f'\n{"="*60}')
    print(f'Training: SateHaze1k {level}')
    print(f'{"="*60}')

    !python train.py \
      --train_input {train_hazy} \
      --train_target {train_gt} \
      --val_input {val_hazy} \
      --val_target {val_gt} \
      --epochs 100 \
      --lr 0.0002 \
      --batch_size_train 8 \
      --batch_size_val 8 \
      --patch_size_train 256 \
      --patch_size_val 256 \
      --val_freq 3 \
      --save_freq 20 \
      --caption \
      --caption_train {cap_train} \
      --caption_val {cap_val} \
      --haze_weight 0.5 \
      --save_dir {save_dir}

In [ ]:
#@title 9b. Train on RRSHID (All Levels)
# Train each RRSHID fog level. Uses the dedicated val split.

for level in RRSHID_LEVELS:
    gt_name_train = find_gt_folder(RRSHID_BASE, level, 'train')
    gt_name_val   = find_gt_folder(RRSHID_BASE, level, 'val')
    if gt_name_train is None or gt_name_val is None:
        print(f'Skipping RRSHID {level}: GT folder not found')
        continue

    train_hazy = f'{RRSHID_BASE}/{level}/train/hazy'
    train_gt   = f'{RRSHID_BASE}/{level}/train/{gt_name_train}'
    val_hazy   = f'{RRSHID_BASE}/{level}/val/hazy'
    val_gt     = f'{RRSHID_BASE}/{level}/val/{gt_name_val}'
    cap_train  = caption_path(train_hazy)
    cap_val    = caption_path(val_hazy)
    save_dir   = f'{RESULTS_DIR}/weights/RRSHID_{level}'
    os.makedirs(save_dir, exist_ok=True)

    if not (os.path.isdir(train_hazy) and os.path.isdir(train_gt)):
        print(f'Skipping {level}: paths not found')
        continue

    print(f'\n{"="*60}')
    print(f'Training: RRSHID {level}')
    print(f'{"="*60}')

    !python train.py \
      --train_input {train_hazy} \
      --train_target {train_gt} \
      --val_input {val_hazy} \
      --val_target {val_gt} \
      --epochs 100 \
      --lr 0.0002 \
      --batch_size_train 8 \
      --batch_size_val 8 \
      --patch_size_train 256 \
      --patch_size_val 256 \
      --val_freq 3 \
      --save_freq 20 \
      --caption \
      --caption_train {cap_train} \
      --caption_val {cap_val} \
      --haze_weight 0.5 \
      --save_dir {save_dir}

---
## Phase 3: Test Trained Weights

Run inference and evaluate the trained models on all fog levels.

In [ ]:
#@title 10. Run Inference with Trained Weights (All Levels)

# --- SateHaze1k ---
for level in SATEHAZE_LEVELS:
    test_hazy  = f'{SATEHAZE_BASE}/{level}/test/hazy'
    test_gt    = f'{SATEHAZE_BASE}/{level}/test/GT'
    cap_test   = caption_path(test_hazy)
    weights    = f'{RESULTS_DIR}/weights/SateHaze1k_{level}/model_best.pth'
    result_dir = f'{RESULTS_DIR}/trained_results/SateHaze1k_{level}'
    os.makedirs(result_dir, exist_ok=True)

    if not os.path.isfile(weights):
        print(f'Skipping SateHaze1k {level}: no trained weights found at {weights}')
        continue

    if os.path.isdir(test_hazy) and os.path.isdir(test_gt):
        print(f'\n=== Inference: SateHaze1k {level} (trained) ===')
        !python infer.py \
          --test_input {test_hazy} \
          --test_target {test_gt} \
          --weights {weights} \
          --caption \
          --caption_test {cap_test} \
          --result_path {result_dir}

# --- RRSHID ---
for level in RRSHID_LEVELS:
    gt_name = find_gt_folder(RRSHID_BASE, level, 'test')
    if gt_name is None:
        continue
    test_hazy  = f'{RRSHID_BASE}/{level}/test/hazy'
    test_gt    = f'{RRSHID_BASE}/{level}/test/{gt_name}'
    cap_test   = caption_path(test_hazy)
    weights    = f'{RESULTS_DIR}/weights/RRSHID_{level}/model_best.pth'
    result_dir = f'{RESULTS_DIR}/trained_results/RRSHID_{level}'
    os.makedirs(result_dir, exist_ok=True)

    if not os.path.isfile(weights):
        print(f'Skipping RRSHID {level}: no trained weights found at {weights}')
        continue

    if os.path.isdir(test_hazy) and os.path.isdir(test_gt):
        print(f'\n=== Inference: RRSHID {level} (trained) ===')
        !python infer.py \
          --test_input {test_hazy} \
          --test_target {test_gt} \
          --weights {weights} \
          --caption \
          --caption_test {cap_test} \
          --result_path {result_dir}

In [ ]:
#@title 11. Evaluate Trained Models (All Levels)

# --- SateHaze1k ---
for level in SATEHAZE_LEVELS:
    result_dir = f'{RESULTS_DIR}/trained_results/SateHaze1k_{level}'
    test_gt    = f'{SATEHAZE_BASE}/{level}/test/GT'
    if os.path.isdir(result_dir) and os.path.isdir(test_gt) and os.listdir(result_dir):
        print(f'\n=== Evaluation: SateHaze1k {level} (trained) ===')
        !python evaluate.py \
          --train_folder {result_dir} \
          --target_folder {test_gt}

# --- RRSHID ---
for level in RRSHID_LEVELS:
    gt_name = find_gt_folder(RRSHID_BASE, level, 'test')
    if gt_name is None:
        continue
    result_dir = f'{RESULTS_DIR}/trained_results/RRSHID_{level}'
    test_gt    = f'{RRSHID_BASE}/{level}/test/{gt_name}'
    if os.path.isdir(result_dir) and os.path.isdir(test_gt) and os.listdir(result_dir):
        print(f'\n=== Evaluation: RRSHID {level} (trained) ===')
        !python evaluate.py \
          --train_folder {result_dir} \
          --target_folder {test_gt}

---
## Phase 4: Resume Training from a Checkpoint

Continue training from a previously saved full-state checkpoint (model +
optimizer + scheduler + epoch + best PSNR). The cosine schedule resumes from
the saved epoch and metrics are appended to `training_metrics.csv`.

**Edit the variables below** to select which dataset/level to resume.

In [ ]:
#@title 12. Resume Training from a Checkpoint
# Edit these variables to select which dataset/level to resume training on.

RESUME_DATASET = 'SateHaze1k'  # 'SateHaze1k' or 'RRSHID'
RESUME_LEVEL   = 'Haze1k_thick'  # e.g. 'Haze1k_thick', 'thick_fog', etc.
RESUME_EPOCHS  = 150  # Total epochs (including already completed ones)

if RESUME_DATASET == 'SateHaze1k':
    train_hazy = f'{SATEHAZE_BASE}/{RESUME_LEVEL}/train/hazy'
    train_gt   = f'{SATEHAZE_BASE}/{RESUME_LEVEL}/train/GT'
    val_hazy   = f'{SATEHAZE_BASE}/{RESUME_LEVEL}/test/hazy'
    val_gt     = f'{SATEHAZE_BASE}/{RESUME_LEVEL}/test/GT'
elif RESUME_DATASET == 'RRSHID':
    gt_name_train = find_gt_folder(RRSHID_BASE, RESUME_LEVEL, 'train')
    gt_name_val   = find_gt_folder(RRSHID_BASE, RESUME_LEVEL, 'val')
    train_hazy = f'{RRSHID_BASE}/{RESUME_LEVEL}/train/hazy'
    train_gt   = f'{RRSHID_BASE}/{RESUME_LEVEL}/train/{gt_name_train}'
    val_hazy   = f'{RRSHID_BASE}/{RESUME_LEVEL}/val/hazy'
    val_gt     = f'{RRSHID_BASE}/{RESUME_LEVEL}/val/{gt_name_val}'

cap_train = caption_path(train_hazy)
cap_val   = caption_path(val_hazy)
save_dir = f'{RESULTS_DIR}/weights/{RESUME_DATASET}_{RESUME_LEVEL}'
resume_path = f'{save_dir}/model_best.pth'

print(f'Resuming {RESUME_DATASET} {RESUME_LEVEL} from: {resume_path}')
print(f'Training to epoch {RESUME_EPOCHS}')

!python train.py \
  --train_input {train_hazy} \
  --train_target {train_gt} \
  --val_input {val_hazy} \
  --val_target {val_gt} \
  --epochs {RESUME_EPOCHS} \
  --lr 0.0002 \
  --batch_size_train 8 \
  --batch_size_val 8 \
  --patch_size_train 256 \
  --patch_size_val 256 \
  --val_freq 3 \
  --save_freq 20 \
  --caption \
  --caption_train {cap_train} \
  --caption_val {cap_val} \
  --haze_weight 0.5 \
  --resume {resume_path} \
  --save_dir {save_dir}